# 01 · Explore the synthetic fab\n\nThe fab spec is deterministic in `fab.seed`. Here we inspect the work centers, the product routes, and the static per-work-center load that hints at the bottleneck.

In [ ]:
import pandas as pd\nfrom fabflow.config import load_config\nfrom fabflow.data.synthetic_fab import build_fab\n\ncfg = load_config('../configs/config.yaml')\nspec = build_fab(cfg.fab, cfg.sim.wafers_per_lot)\n\npd.DataFrame([{'tool_group': t.name, 'n_tools': t.n_tools} for t in spec.tool_groups.values()])

In [ ]:
# products and their routes\nfor p in spec.products.values():\n    print(f'{p.name}: {len(p.route)} steps, RPT={p.raw_process_time:.1f}h, planned CT={p.planned_cycle_time:.1f}h (flow factor {p.planned_cycle_time/p.raw_process_time:.2f})')

In [ ]:
# static load per work center (sum of base proc time across one lot of each product)\nload = spec.tool_group_load()\ndf = pd.DataFrame([{'tool_group': spec.tool_groups[g].name, 'load_h': v, 'n_tools': spec.tool_groups[g].n_tools} for g, v in load.items()])\ndf['load_per_tool'] = df['load_h'] / df['n_tools']\ndf.sort_values('load_per_tool', ascending=False)  # highest = likely bottleneck